In [ ]:
!pip -q install -U "torch>=2.2,<3.0" "torchvision" "torchaudio" "datasets>=3.0.1" "transformers>=4.45.2" "accelerate>=1.0.1" "evaluate>=0.4.2"


import torch, transformers, datasets, evaluate
import numpy as np

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("Transforemrs:", transformers.__version__, "| Datasets:", datasets.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device : {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.8 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128 | CUDA: True
Transforemrs: 5.2.0 | Datasets: 4.0.0
Using device : cuda


In [ ]:
from datasets import load_dataset
beans = load_dataset("beans")
print(beans)

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})


In [ ]:
train = beans['train'].features
test = beans['test'].features
print(train)
print(test)

{'image_file_path': Value('string'), 'image': Image(mode=None, decode=True), 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'])}
{'image_file_path': Value('string'), 'image': Image(mode=None, decode=True), 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'])}


In [ ]:
from transformers import AutoImageProcessor, ViTForImageClassification
import torch

MODEL = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(MODEL)

key ='labels' if 'labels' in beans['train'].features else 'label'
names = beans['train'].features[key].names
id2label = {k : v for k,v in enumerate(names)}
label2id = {v : k for k,v in enumerate(names)}

model = ViTForImageClassification.from_pretrained(
    MODEL,
    num_labels=len(names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)




Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
model.classifier.out_features

3

In [ ]:
import os
num_proc = os.cpu_count()
num_proc

def transform(ex):
  inputs = processor(images=ex['image'],return_tensors='pt')
  ex['pixel_values']= inputs['pixel_values']
  return ex


beans = beans.map(
    transform,
    batched=True,
    remove_columns=['image'],
    num_proc = num_proc,
)

beans.set_format('torch')

Map (num_proc=2):   0%|          | 0/1034 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/133 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/128 [00:00<?, ? examples/s]

In [ ]:
from traitlets.config.loader import ArgumentError
def keep(split):
  cols = ['pixel_values',key] # keep only image and label
  return beans[split].remove_columns([c for c in beans[split].column_names if c not in cols])

train, val, test = keep('train'), keep('validation'),keep('test')

from transformers import TrainingArguments, Trainer, DefaultDataCollator
import evaluate
import numpy as np
import torch

acc = evaluate.load("accuracy")

def metrics(p):
  predictions, labels = p
  pred = np.argmax(predictions,axis=1)
  return {'accuracy' : acc.compute(predictions=pred, references=labels)['accuracy']}

args = TrainingArguments(
   output_dir = '/content/vit_beans',
   eval_strategy='epoch',
   save_strategy='epoch',
   per_device_train_batch_size = 16,
   per_device_eval_batch_size = 32,
   num_train_epochs=3,
   learning_rate = 2e-5,
   fp16 = torch.cuda.is_available(),
   report_to='none',
   remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train,
    eval_dataset=val,
    data_collator=DefaultDataCollator(),
    compute_metrics=metrics
)

trainer.train()
print(trainer.evaluate(test))

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.126960,0.954887
2,No log,0.047641,0.992481
3,No log,0.039495,0.984962


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.08733834326267242, 'eval_accuracy': 0.953125, 'eval_runtime': 2.9022, 'eval_samples_per_second': 44.104, 'eval_steps_per_second': 1.378, 'epoch': 3.0}


In [ ]:
import torch

for i in [0,1]:
    ex = beans['test'][i]

    # 입력 데이터 처리
    input_tensor = ex['pixel_values'].clone().detach()
    inputs = input_tensor.unsqueeze(0).to(model.device)

    # 모델 예측
    with torch.no_grad():
        logits = model(inputs).logits
        pred = logits.argmax(-1).item() # 정수변환

    # 정답 라벨 가져오기
    label_key = 'labels' if 'labels' in ex else 'label'

    true_label_id = ex[label_key].item() #.item() 사용하면 tensor(0) >> 0(정수)

    # 결과 출력
    print(f'[{i} 예측: {model.config.id2label[pred]} | {model.config.id2label[true_label_id]}]')



[0 예측: angular_leaf_spot | angular_leaf_spot]
[1 예측: angular_leaf_spot | angular_leaf_spot]
